# Pipeline B: Health Trajectory Model

**Object Name:** Longitudinal Health Deterioration Predictor

**Goal:** Predict if a resident's `general_health_score` will drop below a critical threshold in the next 60 days.

**Business Problem:** Physical health and baseline mental well-being often decline before behavioral incidents occur. By identifying "declining trajectories" in health scores, caseworkers can intervene with medical or counseling support before the resident reaches a crisis point.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Setup Data Paths
REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"

def load_table(name):
    fp = DATA_DIR / f"{name}.csv"
    return pd.read_csv(fp) if fp.exists() else pd.DataFrame()

health = load_table("health_wellbeing_records")
recordings = load_table("process_recordings")

## Feature Engineering (Panel Data / Lags)

We build a dataset where each row is a (resident, month). Features include health scores from 1 and 2 months ago.

In [ ]:
health['assessment_date'] = pd.to_datetime(health['assessment_date'])
health['month_year'] = health['assessment_date'].dt.to_period('M')

# Aggregate to monthly health stats
monthly_health = health.groupby(['resident_id', 'month_year']).agg({
    'general_health_score': 'mean',
    'sleep_score': 'mean',
    'nutrition_score': 'mean',
    'weight_kg': 'last'
}).reset_index()

# Add Lags
monthly_health['health_lag1'] = monthly_health.groupby('resident_id')['general_health_score'].shift(1)
monthly_health['health_lag2'] = monthly_health.groupby('resident_id')['general_health_score'].shift(2)
monthly_health['health_diff'] = monthly_health['health_lag1'] - monthly_health['health_lag2']

# Target: Health drops below 5 (on a 10pt scale) in the NEXT month
monthly_health['future_health'] = monthly_health.groupby('resident_id')['general_health_score'].shift(-1)
monthly_health['will_deteriorate'] = (monthly_health['future_health'] < 5).astype(int)

df = monthly_health.dropna(subset=['health_lag2', 'will_deteriorate'])

## Explanatory Model: Ridge Regression

Identify which health dimensions (sleep, nutrition, etc.) most strongly associate with current overall health deterioration.

In [ ]:
X_vals = df[['sleep_score', 'nutrition_score', 'weight_kg', 'health_diff']]
y_vals = df['general_health_score']

ridge = Pipeline([('scaler', StandardScaler()), ('clf', Ridge())])
ridge.fit(X_vals, y_vals)

coef_summary = pd.Series(ridge.named_steps['clf'].coef_, index=X_vals.columns)
print("Health Dimension Importance (Marginal Lift):")
print(coef_summary.sort_values(ascending=False))

## Predictive Model: Logistic Regression

Predict binary deterioration (True/False).

In [ ]:
X = df[['health_lag1', 'health_lag2', 'health_diff', 'sleep_score']]
y = df['will_deteriorate']

log_reg = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced'))])
log_reg.fit(X, y)

print("Predictive AUC:", roc_auc_score(y, log_reg.predict_proba(X)[:, 1]))